## Variational Autoencoder 

In [2]:
print('connected to kernel')

connected to kernel


**Inserting Libraries**

In [3]:
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt
import os

In [6]:
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
import keras
from keras import ops
from keras import layers
os.environ["KERAS_BACKEND"] = "tensorflow"


**Loading the File and Identifying its contents**

In [7]:
root_file_name = "/Users/ruthhodgson/geant4_apps/QEPET/build/out.root"

root_file = uproot.open(root_file_name)

trees = root_file.keys()
print(list(enumerate(trees)))

tree = root_file[trees[11]] #this corresponds to threeG NCS
info = tree.keys()
print(info)

[(0, 'TwoG_NCS;1'), (1, 'TwoG_PhantomSCS;1'), (2, 'TwoG_ScannerSCS;1'), (3, 'TwoG_PhantomSCS_ScannerSCS;1'), (4, 'TwoG_PhantomDCS;1'), (5, 'TwoG_ScannerDCS;1'), (6, 'TwoG_PhantomSCS_ScannerDCS;1'), (7, 'TwoG_PhantomDCS_ScannerSCS;1'), (8, 'TwoG_PhantomDCS_ScannerDCS;1'), (9, 'TwoG_MCS;1'), (10, 'TwoG_Excluded;1'), (11, 'ThreeG_NCS;1'), (12, 'ThreeG_ScannerSCS;1'), (13, 'ThreeG_ScannerDCS;1'), (14, 'ThreeG_ScannerTCS;1'), (15, 'ThreeG_MCS;1'), (16, 'ThreeG_Excluded;1'), (17, 'Run;1')]
['EventID', 'NGammas', 'AnnihilX_mm', 'AnnihilY_mm', 'AnnihilZ_mm', 'GammaIndex', 'GammaTrackID', 'GammaParentID', 'HitIndex', 'VolumeID', 'VolumeName', 'ProcessID', 'ProcessName', 'EnergyDeposit_keV', 'PreEnergy_keV', 'PostEnergy_keV', 'X_mm', 'Y_mm', 'Z_mm', 'Theta_deg', 'Phi_deg', 'DeltaPhi01_deg', 'DeltaPhi02_deg', 'DeltaPhi12_deg']


**Convert this data into a dataframe**

In [8]:
df = tree.arrays(info, library = 'pd')
print(df[['AnnihilX_mm', 'AnnihilY_mm', 'AnnihilZ_mm', 'X_mm', "Y_mm", "Z_mm"]].head(1))

#num of detector hits = number of events / 3

print()

print("Number of detector hits (this is the number of detections) =", len(df))

print()

# num = file['Run'].arrays('NumberOfEvents') 
print("Number of events (this is the number of annihilations) =",
      df['EventID'].drop_duplicates().shape[0])


   AnnihilX_mm  AnnihilY_mm  AnnihilZ_mm       X_mm       Y_mm       Z_mm
0    -0.293905     4.053828    -8.090448  35.348804 -11.452375 -38.762901

Number of detector hits (this is the number of detections) = 14172

Number of events (this is the number of annihilations) = 4724


In [9]:
detects = []
targets = []

annils = df.groupby('EventID', sort = False)
for i , (event_id, event) in enumerate(annils):
    target = [
        event['AnnihilX_mm'].iloc[0],
        event['AnnihilY_mm'].iloc[0],
        event['AnnihilZ_mm'].iloc[0]
    ]

    x1_detect = event['X_mm'].iloc[0]
    x2_detect = event['X_mm'].iloc[1]
    x3_detect = event['X_mm'].iloc[2]

    y1_detect = event['Y_mm'].iloc[0]
    y2_detect = event['Y_mm'].iloc[1]
    y3_detect = event['Y_mm'].iloc[2]

    z1_detect = event['Z_mm'].iloc[0]
    z2_detect = event['Z_mm'].iloc[1]
    z3_detect = event['Z_mm'].iloc[2]

    detect = [
        [x1_detect, y1_detect, z1_detect], 
        [x2_detect, y2_detect, z2_detect], 
        [x3_detect, y3_detect, z3_detect]
    ]

    detects.append(detect)
    targets.append(target)

#convert to arrays

detects = np.array(detects, dtype=np.float16)
targets = np.array(targets, dtype=np.float32)

print(f'Detections shape = {detects.shape}')
print(f'Targets shape = {targets.shape}')


Detections shape = (4724, 3, 3)
Targets shape = (4724, 3)


In [10]:
#print examples to test the data has been set up correctly
print('Correct Annihilations')
print(df[['AnnihilX_mm', 'AnnihilY_mm', 'AnnihilZ_mm']].head(1))
print('Correct Detections')
print(df[['X_mm', "Y_mm", "Z_mm"]].head(3))

print("\nExample Annihilations:")
print(targets[0])
print('Example Detections')
print(detects[0])


Correct Annihilations
   AnnihilX_mm  AnnihilY_mm  AnnihilZ_mm
0    -0.293905     4.053828    -8.090448
Correct Detections
        X_mm       Y_mm       Z_mm
0  35.348804 -11.452375 -38.762901
1  17.195129 -35.946274  32.120178
2 -44.458641  39.904507   1.532497

Example Annihilations:
[-0.29390496  4.0538282  -8.090448  ]
Example Detections
[[ 35.34  -11.45  -38.75 ]
 [ 17.19  -35.94   32.12 ]
 [-44.47   39.9     1.532]]


In [11]:

input_train, input_test, out_train, out_test = train_test_split(
    detects, targets, test_size = 0.2, random_state = 42
)

print(input_train.shape)
print(input_test.shape)
print(out_train.shape)
print(out_test.shape)

(3779, 3, 3)
(945, 3, 3)
(3779, 3)
(945, 3)


In [49]:
#split into batches and scale output

my_batch_size = 16


train_dataset = (
    tf.data.Dataset.from_tensor_slices(
        (input_train, out_train)
    )
    .shuffle(len(input_train))
    .batch(my_batch_size)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices(
        (input_test, out_test)
    )
    .batch(my_batch_size)
)


**Building the Vae**

In [64]:
#start with the sampling layer
#the call function gives you the variables needed to define the loss function later in the model

latent_dim = 32

class Sampling(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.seed_generator = keras.random.SeedGenerator(annils.ngroups)

    def call(self, inputs):
        z_mean , z_log_var = inputs
        batch = ops.shape(z_mean)[0]
        dimension = ops.shape(z_mean)[1]
        epsilon = keras.random.normal(shape=(batch, dimension), seed = self.seed_generator)
        return z_mean + ops.exp(.5*z_log_var)*epsilon

#encoding layer

encode_inputs = keras.Input(shape=(3, 3))


input = layers.Dense(16, activation= 'relu')(encode_inputs)
input = layers.Dense(16, activation = 'relu')(input)

input = layers.GlobalAveragePooling1D()(input)

input = layers.Dense(16, activation = 'relu')(input)


z_mean = layers.Dense(latent_dim, name = 'z_mean')(input)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(input)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encode_inputs, [z_mean, z_log_var, z], name = 'encoder')
encoder.summary()




Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_13      │ (None, 3, 3)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 3, 16)     │         64 │ input_layer_13[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 3, 16)     │        272 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 16)        │          0 │ dense_15[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 16)        │        272 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_mean (Dense)      │ (None, 32)        │        544 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_log_var (Dense)   │ (None, 32)        │        544 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sampling_6          │ (None, 32)        │          0 │ z_mean[0][0],     │
│ (Sampling)          │                   │            │ z_log_var[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,696 (6.62 KB)

 Trainable params: 1,696 (6.62 KB)

 Non-trainable params: 0 (0.00 B)

In [69]:
#decoder 

latent_inputs = keras.Input(shape=( latent_dim,))
input=layers.Dense(16, activation='relu')(latent_inputs)
input=layers.Dense(16, activation='relu')(input)


decode_outputs = layers.Dense(3)(input)
decoder = keras.Model(latent_inputs, decode_outputs, name = 'decoder')
decoder.summary()

Model: "decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_16 (InputLayer)     │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 851 (3.32 KB)

 Trainable params: 851 (3.32 KB)

 Non-trainable params: 0 (0.00 B)

**VAE**

In [70]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder= encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name = 'total_loss')
        self.learning_loss_tracker = keras.metrics.Mean(name = "learning_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name='kl_loss')
    @property
    def metrics(self):
        return[
            self.total_loss_tracker,
            self.learning_loss_tracker,
            self.kl_loss_tracker
        ]

    def train_step(self, data):
        hits, anns = data
        with tf.GradientTape() as tape:
            z_mean, z_log_var , z = self.encoder(hits)
            learning_output = self.decoder(z)
            learning_loss = ops.mean(ops.square(learning_output- anns))
                
            kl_loss = -0.5*(1+z_log_var - ops.square(z_mean)-ops.exp(z_log_var))
            kl_loss = ops.mean(ops.sum(kl_loss, axis = 1))

            loss = learning_loss + kl_loss
        
        gradients = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))
        self.total_loss_tracker.update_state(loss)
        self.learning_loss_tracker.update_state(learning_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss":self.total_loss_tracker.result(),
            "learning_loss":self.learning_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result()
        }


**Training this vae**

In [52]:
print(type(input_train), type(out_train))

<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [72]:
# input_data = np.concatenate([input_train, input_test], axis = 0)

vae = VAE(encoder, decoder)
vae.compile(optimizer= keras.optimizers.Adam())
vae.fit(train_dataset, epochs = 50, batch_size = my_batch_size)

Epoch 1/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 2s 807us/step - kl_loss: 1.0900 - learning_loss: 4.9194 - loss: 6.0093
Epoch 2/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 770us/step - kl_loss: 0.1770 - learning_loss: 4.7859 - loss: 4.9629
Epoch 3/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 768us/step - kl_loss: 0.0836 - learning_loss: 4.7573 - loss: 4.8410
Epoch 4/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 779us/step - kl_loss: 0.0485 - learning_loss: 4.7726 - loss: 4.8212
Epoch 5/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 769us/step - kl_loss: 0.0358 - learning_loss: 4.7304 - loss: 4.7662
Epoch 6/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 815us/step - kl_loss: 0.0226 - learning_loss: 4.7255 - loss: 4.7482
Epoch 7/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 777us/step - kl_loss: 0.0185 - learning_loss: 4.7179 - loss: 4.7363
Epoch 8/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 769us/step - kl_loss: 0.0131 - learning_loss: 4.7053 - loss: 4.7184
Epoch 9/50
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 777us/step - kl_loss: 0.0102 - learning_loss: 4.6948 - loss: 4.7051
E